# einops-repeat-broadcast — ex2: per-token positional bias broadcast across batch

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat-broadcast`. Running the final beacon cell reports progress against the `Einops: Repeat-as-broadcast` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat-as-broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat-broadcast`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat-broadcast"
DD_SUBTOPIC = "Einops: Repeat-as-broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Einops repeat as broadcast — quick refresher

`einops.repeat(x, '... -> ... new', new=N)` is **not** a memcpy when used for the pure broadcast pattern. Under the hood, einops compiles it to a `torch.Tensor.expand` (or numpy stride trick) that sets the **stride of the new axis to zero** — every position along the new axis points at the same underlying storage cell.

**Three names for the same trick:**
- `einops.repeat(x, 'b d -> b n d', n=N)`
- `x.unsqueeze(1).expand(-1, N, -1)`
- `x[:, None, :].broadcast_to((x.shape[0], N, x.shape[1]))`

All three produce a view with `stride=0` on the inserted axis. No memory is allocated for the duplicates — they're a single value read N times.

**When this is the right call.** Anywhere you need to *pair every X with every Y* (e.g. every ray with every triangle in ARENA's ray tracer), reach for `einops.repeat` — it produces the expanded view in O(1) memory.

**Compared to `repeat` that actually copies.** `einops.repeat(x, 'b d -> b (n d)', n=N)` *does* materialise the copy because the output shape collapses the repeat axis into another. Only patterns that **insert** a new axis (and leave it un-grouped) stay at stride 0.

### Exercise 2 — per-token positional bias broadcast across batch

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `einops.repeat` to broadcast a `(T,)` per-position scalar bias across an `(B, H, T)` batch axis tensor without copying — the canonical transformer per-token bias pattern.
> Keywords: broadcast, positional-bias, attention, transformer
> ```

**KCs targeted:** `repeat-inserts-zero-stride-axis`, `repeat-vs-torch-method-repeat`

Implement `ex2_broadcast_bias(bias, scores)`.

- `bias` has shape `(T,)` — one scalar per sequence position.
- `scores` has shape `(B, H, T)` — per-head per-token attention logits.

Return `bias_b` with shape `(B, H, T)` — `bias` broadcast across the batch and head axes so it can be *added* to `scores`.

**Constraint — no copy.** Use `einops.repeat` to insert the leading `b` and `h` axes. The returned tensor must share storage with `bias` (the test asserts `data_ptr()` equality and stride-0 on the inserted axes).

Pattern: `einops.repeat(bias, 't -> b h t', b=B, h=H)`.

**Don't return `scores + bias_b`** — just return `bias_b`. The drill is about producing the broadcast view, not the addition.

In [ ]:
def ex2_broadcast_bias(bias: Tensor, scores: Tensor) -> Tensor:
    B, H, T = scores.shape
    return einops.repeat(bias, 't -> b h t', b=B, h=H)


<details><summary>Solution</summary>

```python
def ex2_broadcast_bias(bias: Tensor, scores: Tensor) -> Tensor:
    B, H, T = scores.shape
    return einops.repeat(bias, 't -> b h t', b=B, h=H)
```

**Why this matters more than it looks.** In a real transformer the bias broadcast happens at every layer, every step — if it materialised, you'd pay `B * H * T` memory per layer (which for `B=8, H=32, T=2048` is half a million floats). Keeping it as a stride-0 view costs `T` floats total.

**`einops.repeat` vs `torch.Tensor.repeat`.** Same name, opposite semantics! `bias.repeat(B, H, 1)` *does* memcpy — `torch.Tensor.repeat` is the materialising version. `einops.repeat` is the broadcast-when-possible version. Confusing, but the storage assertion in the test catches it instantly.

**Equivalent torch idiom.** `bias.view(1, 1, T).expand(B, H, T)` is the same view, written in raw PyTorch. einops is preferred because the pattern string `'t -> b h t'` documents the dimensional intent.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()